# 02 — Model Development

I am bringing together the comparisons from Weeks 5–8 using the same three reduced predictors. Recall for high-volatility days remains the main metric. Precision and F1 help me see the cost of false alarms.

These are the same 2024 and 2025 periods I already used for development, so the results are not a new independent test. I do not download, fit on, or evaluate any 2026 data here.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Find the shared code from the project folder or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif (project_root / "ANLY530_SPY_Volatility_Project").is_dir():
    project_root = project_root / "ANLY530_SPY_Volatility_Project"
sys.path.insert(0, str(project_root))

from src.data_utils import download_spy, create_features, create_target, FEATURES, EVALUATION_YEARS

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score, f1_score

In [2]:
prices = download_spy()
feature_data = create_features(prices)
prepared_data, cutoff = create_target(feature_data)
data = prepared_data.dropna(subset=FEATURES + ["target", "target_date"]).copy()
data["target"] = data["target"].astype(int)

print(f"Price dates: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Fixed 2020–2023 volatility cutoff: {cutoff:.4%}")
print("Predictors:", FEATURES)
assert prices.index.max() < pd.Timestamp("2026-01-01")
assert data["target_date"].max() < pd.Timestamp("2026-01-01")

Price dates: 2020-01-02 to 2025-12-31
Fixed 2020–2023 volatility cutoff: 1.3905%
Predictors: ['daily_return', 'average_return_5d', 'volatility_20d']


## Keep the earlier model settings

Logistic regression is the baseline from Week 5. Week 7 added kNN with k = 3, 5, and 7 using Euclidean distance, plus Gaussian Naive Bayes. Week 8 added decision trees with maximum depths of 2, 3, 4, and 5. I keep those settings without adding another search.

Each pipeline fits its scaler using only that period's training data. Scaling matters for kNN distances and logistic regression; trees do not need it, but I keep the Week 8 pipeline unchanged. The fuller L1 experiment from Week 6 stays in that notebook because it used seven predictors.

In [3]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression())
}

for k in [3, 5, 7]:
    models[f"kNN (k={k})"] = make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    )

models["Gaussian Naive Bayes"] = make_pipeline(StandardScaler(), GaussianNB())

for depth in [2, 3, 4, 5]:
    models[f"Decision Tree depth {depth}"] = make_pipeline(
        StandardScaler(), DecisionTreeClassifier(max_depth=depth, random_state=42)
    )

## Evaluate in time order

I first train on outcomes through 2023 and evaluate 2024. Then I include 2024 in training and evaluate 2025. The split uses `target_date`, so the outcome for every training row is known before the evaluation year starts.

Every model uses the same rows, predictors, target, and years. I refit each pipeline for each period without shuffling. The models keep their usual prediction rules; I do not adjust a classification threshold.

In [4]:
result_rows = []
window_rows = []
tree_importances = {}

for year in EVALUATION_YEARS:
    train = data[data["target_date"].dt.year < year]
    test = data[data["target_date"].dt.year == year]
    assert train["target_date"].max() < test["target_date"].min()

    window_rows.append({
        "Evaluation Period": year,
        "Last training outcome": train["target_date"].max().date(),
        "Training rows": len(train),
        "Evaluation rows": len(test),
        "High-volatility outcomes": int(test["target"].sum())
    })

    for name, model in models.items():
        model.fit(train[FEATURES], train["target"])
        predictions = model.predict(test[FEATURES])
        result_rows.append({
            "Evaluation Period": year,
            "Model": name,
            "Recall": recall_score(test["target"], predictions, pos_label=1),
            "Precision": precision_score(test["target"], predictions, pos_label=1, zero_division=0),
            "F1-score": f1_score(test["target"], predictions, pos_label=1, zero_division=0)
        })

        if name == "Decision Tree depth 2":
            tree_importances[f"Train through {year - 1}"] = model.named_steps["decisiontreeclassifier"].feature_importances_.copy()

windows = pd.DataFrame(window_rows)
results = pd.DataFrame(result_rows)
display(windows)

,Evaluation Period,Last training outcome,Training rows,Evaluation rows,High-volatility outcomes
0,2024,2023-12-29,985,252,4
1,2025,2024-12-31,1237,250,28


In [5]:
results_table = results.copy()
for column in ["Recall", "Precision", "F1-score"]:
    results_table[column] = results_table[column].map("{:.2%}".format)
display(results_table)

,Evaluation Period,Model,Recall,Precision,F1-score
0,2024,Logistic Regression,0.00%,0.00%,0.00%
1,2024,kNN (k=3),25.00%,20.00%,22.22%
2,2024,kNN (k=5),0.00%,0.00%,0.00%
3,2024,kNN (k=7),0.00%,0.00%,0.00%
4,2024,Gaussian Naive Bayes,0.00%,0.00%,0.00%
5,2024,Decision Tree depth 2,75.00%,60.00%,66.67%
6,2024,Decision Tree depth 3,50.00%,66.67%,57.14%
7,2024,Decision Tree depth 4,50.00%,66.67%,57.14%
8,2024,Decision Tree depth 5,0.00%,0.00%,0.00%
9,2025,Logistic Regression,85.71%,96.00%,90.57%


All three metrics refer to high volatility, class 1. Precision is undefined when there are no high-volatility predictions; `zero_division=0` displays it as zero. A zero recall means that the model detected none of the actual high-volatility outcomes in that period.

I also keep the Week 5 check of recall variation across years. Each year has equal weight, and the SD is the sample standard deviation across the two years. It is not a confidence interval.

In [6]:
recall_summary = results.groupby("Model", sort=False)["Recall"].agg(["mean", "min", "max", "std"])
recall_summary.columns = ["Mean recall", "Minimum recall", "Maximum recall", "Recall SD"]
display(recall_summary.round(4))

,Mean recall,Minimum recall,Maximum recall,Recall SD
Model,,,,
Logistic Regression,0.4286,0.00,0.8571,0.6061
kNN (k=3),0.5893,0.25,0.9286,0.4798
kNN (k=5),0.4464,0.00,0.8929,0.6313
kNN (k=7),0.4464,0.00,0.8929,0.6313
Gaussian Naive Bayes,0.4464,0.00,0.8929,0.6313
Decision Tree depth 2,0.8393,0.75,0.9286,0.1263
Decision Tree depth 3,0.6964,0.50,0.8929,0.2778
Decision Tree depth 4,0.6964,0.50,0.8929,0.2778
Decision Tree depth 5,0.4464,0.00,0.8929,0.6313


## Keep the Week 8 interpretation check

Week 8 identified the depth-2 tree as a candidate for later testing. I show its feature importance from each training period again. These values describe the fitted tree's splits; they do not show causal effects or prove that a feature will stay useful in a later market period.

In [7]:
importance_table = pd.DataFrame(tree_importances, index=FEATURES)
importance_table.index.name = "Feature"
display(importance_table.round(4))

# Summarize the actual results for the earlier candidate and baseline.
for name in ["Logistic Regression", "kNN (k=3)", "Decision Tree depth 2"]:
    model_results = results[results["Model"] == name].set_index("Evaluation Period")
    print(f"{name}: recall was {model_results.loc[2024, 'Recall']:.2%} in 2024 "
          f"and {model_results.loc[2025, 'Recall']:.2%} in 2025.")

,Train through 2023,Train through 2024
Feature,,
daily_return,0.0,0.0
average_return_5d,0.0,0.0
volatility_20d,1.0,1.0


Logistic Regression: recall was 0.00% in 2024 and 85.71% in 2025.
kNN (k=3): recall was 25.00% in 2024 and 92.86% in 2025.
Decision Tree depth 2: recall was 75.00% in 2024 and 92.86% in 2025.


## My next decision

I will use these development results to review the recall and false-alarm tradeoff before fixing a final model. The depth-2 tree remains the candidate carried forward from Week 8; this notebook does not automatically select a new winner or claim it will perform best in 2026.

One limitation is that these same evaluation years have been used repeatedly to compare methods. Today's and tomorrow's volatility also share 19 returns, so the results partly reflect persistence. Before opening the later holdout, I need to record the final model settings and the exact evaluation dates in notebook 03.